# 04zz — Cleanup del master de gustos

3 versiones progresivas: v1_conservative, v2_intermediate, v3_aggressive.
Sin TARGET_LEAKAGE (no hay target predictivo). Cleanup puramente estadístico:
high-NaN, low-variance, high-correlation.

In [ ]:
import pandas as pd
import numpy as np
import time, json
from pathlib import Path
from datetime import datetime

ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase4_gustos' else Path.cwd()
DATA_QC = ROOT / 'data' / 'data_qc_gustos'
INFORMES_DIR = ROOT / 'informes' / 'fase4_gustos' / '03_master'
REPORT_CLEANUP = INFORMES_DIR / '04zz_master_cleanup'
REPORT_CLEANUP.mkdir(parents=True, exist_ok=True)

master = pd.read_parquet(DATA_QC / 'master_table_gustos.parquet')
N = len(master)
N_FEATURES_INIT = master.shape[1] - 1  # excluyendo user_id
print(f'Master cargado: {master.shape}')

CLEANUP_CONFIG = {
    'v1_conservative': {'max_nan_pct': 0.95, 'min_variance_threshold': 0.0001, 'max_correlation': None},
    'v2_intermediate': {'max_nan_pct': 0.90, 'min_variance_threshold': 0.001, 'max_correlation': 0.99},
    'v3_aggressive':   {'max_nan_pct': 0.85, 'min_variance_threshold': 0.005, 'max_correlation': 0.95},
}


In [ ]:
def apply_cleanup(master_df, config, version_name):
    t0 = time.time()
    df = master_df.copy()
    drops_log = {'high_nan': [], 'low_variance': [], 'high_corr': []}

    feature_cols = [c for c in df.columns if c != 'user_id']

    # 1. High-NaN
    nan_rates = df[feature_cols].isna().mean()
    high_nan = nan_rates[nan_rates > config['max_nan_pct']].index.tolist()
    drops_log['high_nan'] = high_nan
    df = df.drop(columns=high_nan)
    print(f'  [{version_name}] drop high_nan: {len(high_nan)}')

    # 2. Low-variance (solo numéricas)
    feature_cols = [c for c in df.columns if c != 'user_id']
    numeric_cols = df[feature_cols].select_dtypes(include=['number']).columns.tolist()
    variances = df[numeric_cols].var()
    low_var = variances[variances < config['min_variance_threshold']].index.tolist()
    drops_log['low_variance'] = low_var
    df = df.drop(columns=low_var)
    print(f'  [{version_name}] drop low_variance: {len(low_var)}')

    # 3. High-correlation pairs (si max_correlation está definido)
    if config['max_correlation'] is not None:
        feature_cols = [c for c in df.columns if c != 'user_id']
        numeric_cols = df[feature_cols].select_dtypes(include=['number']).columns.tolist()
        if len(numeric_cols) > 1:
            corr_matrix = df[numeric_cols].corr().abs()
            upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
            high_corr = [c for c in upper.columns if any(upper[c].dropna() >= config['max_correlation'])]
            drops_log['high_corr'] = high_corr
            df = df.drop(columns=high_corr)
            print(f'  [{version_name}] drop high_corr (>={config["max_correlation"]}): {len(high_corr)}')

    elapsed = time.time() - t0
    return df, drops_log, elapsed

print('apply_cleanup definida')


In [ ]:
# Aplicar las 3 versiones
results = {}
all_drops = {}
for version_name, config in CLEANUP_CONFIG.items():
    print(f'\n=== {version_name} ===')
    df_v, drops, elapsed = apply_cleanup(master, config, version_name)
    results[version_name] = df_v
    all_drops[version_name] = drops
    out_path = DATA_QC / f'master_table_gustos_{version_name}.parquet'
    df_v.to_parquet(out_path, index=False)
    n_final = df_v.shape[1] - 1  # excluyendo user_id
    n_dropped = N_FEATURES_INIT - n_final
    print(f'  → guardado: {df_v.shape} (n_features={n_final}, dropped={n_dropped}) en {elapsed:.1f}s')

print('\nLas 3 versiones generadas')


In [ ]:
# Reporte ejecutivo 04zz
lines = [
    '# 04zz — Cleanup del master de gustos',
    '',
    f'**Fecha**: {datetime.now().strftime("%Y-%m-%d %H:%M")}',
    f'**Master inicial**: {master.shape} ({N_FEATURES_INIT} features + user_id)',
    '',
    '## Resumen por versión',
    '',
    '| Versión | n_inicial | drop high_nan | drop low_var | drop high_corr | n_final |',
    '|---|---:|---:|---:|---:|---:|',
]
for v, drops in all_drops.items():
    n_final = results[v].shape[1] - 1
    lines.append(f'| {v} | {N_FEATURES_INIT} | {len(drops["high_nan"])} | {len(drops["low_variance"])} | {len(drops["high_corr"])} | {n_final} |')

lines += ['', '## Configuración aplicada', '']
for v, cfg in CLEANUP_CONFIG.items():
    lines += [
        f'### {v}',
        '',
        f'- `max_nan_pct`: {cfg["max_nan_pct"]}',
        f'- `min_variance_threshold`: {cfg["min_variance_threshold"]}',
        f'- `max_correlation`: {cfg["max_correlation"]}',
        '',
    ]

lines += ['## Features eliminadas por versión y motivo', '']
for v, drops in all_drops.items():
    lines += [f'### {v}', '']
    for category, cols in drops.items():
        if cols:
            lines += [f'**{category}** ({len(cols)}):', '']
            for c in sorted(cols):
                lines.append(f'- `{c}`')
            lines.append('')

(REPORT_CLEANUP / 'execution_report.md').write_text('\n'.join(lines))
print('execution_report.md guardado')


In [ ]:
# feature_quality_report.md — análisis cualitativo
feature_cols = [c for c in master.columns if c != 'user_id']
nan_rates = (master[feature_cols].isna().mean()*100).round(2)
numeric_cols = master[feature_cols].select_dtypes(include=['number']).columns.tolist()
variances = master[numeric_cols].var().sort_values(ascending=False)

# Origen de cada feature
from pathlib import Path as _P
DATA_QC = _P('data/data_qc_gustos')
PARQUETS = {
    'reutilizadas_churn': pd.read_parquet(DATA_QC / 'features_reutilizadas_churn.parquet').columns.tolist(),
    'proporciones': pd.read_parquet(DATA_QC / 'features_proporciones.parquet').columns.tolist(),
    'diversidad': pd.read_parquet(DATA_QC / 'features_diversidad.parquet').columns.tolist(),
    'temporales': pd.read_parquet(DATA_QC / 'features_temporales.parquet').columns.tolist(),
    'intensidad': pd.read_parquet(DATA_QC / 'features_intensidad.parquet').columns.tolist(),
    'monetizacion': pd.read_parquet(DATA_QC / 'features_monetizacion.parquet').columns.tolist(),
}
feature_origin = {}
for origin, cols in PARQUETS.items():
    for c in cols:
        if c == 'user_id': continue
        if c not in feature_origin:
            feature_origin[c] = origin

origin_counts = {}
for o in feature_origin.values():
    origin_counts[o] = origin_counts.get(o, 0) + 1

# Top correlation pairs (>0.90)
if len(numeric_cols) > 1:
    corr = master[numeric_cols].corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    pairs = upper.stack().reset_index()
    pairs.columns = ['feat_a', 'feat_b', 'abs_corr']
    high_pairs = pairs[pairs['abs_corr'] >= 0.90].sort_values('abs_corr', ascending=False).head(30)

lines = [
    '# Feature quality report (gustos master)',
    '',
    f'**Fecha**: {datetime.now().strftime("%Y-%m-%d %H:%M")}',
    f'**Master**: {master.shape}',
    '',
    '## Distribución por origen (bloque de feature engineering)',
    '',
    '| Origen | n_features |',
    '|---|---:|',
]
for o, n in sorted(origin_counts.items(), key=lambda x: -x[1]):
    lines.append(f'| {o} | {n} |')

lines += ['', '## Top 15 features con más NaN', '', '| Feature | Origen | %NaN |', '|---|---|---:|']
for f, v in nan_rates.sort_values(ascending=False).head(15).items():
    lines.append(f'| {f} | {feature_origin.get(f, "?")} | {v} |')

lines += ['', '## Top 15 features por varianza (numéricas)', '', '| Feature | Origen | Varianza |', '|---|---|---:|']
for f in variances.head(15).index:
    lines.append(f'| {f} | {feature_origin.get(f, "?")} | {variances[f]:.4g} |')

lines += ['', '## Bottom 15 features por varianza (cuasi-constantes)', '', '| Feature | Origen | Varianza |', '|---|---|---:|']
for f in variances.tail(15).index:
    lines.append(f'| {f} | {feature_origin.get(f, "?")} | {variances[f]:.4g} |')

if 'high_pairs' in dir() and len(high_pairs) > 0:
    lines += ['', '## Top 30 pares con correlación >= 0.90', '', '| Feature A | Feature B | abs(corr) |', '|---|---|---:|']
    for _, row in high_pairs.iterrows():
        lines.append(f'| {row["feat_a"]} | {row["feat_b"]} | {row["abs_corr"]:.4f} |')

(REPORT_CLEANUP.parent / 'feature_quality_report.md').write_text('\n'.join(lines))
print('feature_quality_report.md guardado')
